<a href="https://colab.research.google.com/github/ShivenSiwach/Imbalanced_dataset_by_SMOTE/blob/main/Imbalanced_dataset_by_SMOTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CVD Prediction with Resampling (SMOTE, ADASYN, Borderline-SMOTE)
# Reproduces the type of tables and confusion matrices from the dissertation.


# 1. IMPORT LIBRARIES

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE

# For pretty printing tables (optional, but nice)
try:
    from tabulate import tabulate
    TABULATE_AVAILABLE = True
except ImportError:
    TABULATE_AVAILABLE = False


# 2. CONFIG: PATH & PARAMETERS
# TODO: Change this to your real dataset file path
DATA_PATH = "/content/Z-Alizadeh sani dataset (2).csv"

# Name of target/label column
TARGET_COL = "Cath"

TEST_SIZE = 0.2
RANDOM_STATE = 42


# 3. HELPER: LOAD & BASIC PREPROCESSING

def load_dataset(path):
    """
    Load the Z-Alizadeh Sani dataset from CSV.
    Assumes the target column is TARGET_COL.
    """
    df = pd.read_csv(path)
    print("Dataset loaded. Shape:", df.shape)
    return df


def split_features_target(df, target_col):
    """
    Split dataframe into X (features) and y (target).
    """
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # If target is not already binary 0/1, you may need to map it.
    y = y.map({'Cad': 1}).fillna(0).astype(int)
    return X, y


def get_column_types(X):
    """
    Separate numeric and categorical feature names.
    This will be used in ColumnTransformer.
    """
    numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    return numeric_cols, categorical_cols


def build_preprocessor(numeric_cols, categorical_cols):
    """
    Build a ColumnTransformer that:
    - Scales numeric columns (StandardScaler)
    - One-hot encodes categorical columns
    """
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown="ignore")

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols)
        ]
    )
    return preprocessor


# 4. DEFINE ML MODELS
def get_models():
    """
    Define and return dictionary of ML models:
    Random Forest, SVM, ANN (MLP), Logistic Regression
    """
    models = {
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            min_samples_split=2,
            random_state=RANDOM_STATE
        ),
        "SVM": SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,  # needed for AUC
            random_state=RANDOM_STATE
        ),
        "ANN": MLPClassifier(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="adam",
            learning_rate_init=0.001,
            batch_size=16,
            max_iter=500,
            random_state=RANDOM_STATE
        ),
        "LogisticRegression": LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    }
    return models


# 5. METRIC CALCULATION FUNCTION
def evaluate_model(clf, X_test, y_test, model_name="", resampling_name="None"):
    """
    Compute metrics and confusion matrix for a trained model.
    Returns:
        metrics_dict, cm
    """
    y_pred = clf.predict(X_test)

    # For AUC, we need probabilities or decision function
    if hasattr(clf, "predict_proba"):
        y_proba = clf.predict_proba(X_test)[:, 1]
    else:
        try:
            y_proba = clf.decision_function(X_test)
        except Exception:
            # If no prob or decision_function, use y_pred as a fallback (AUC might not be meaningful)
            y_proba = y_pred

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba)

    cm = confusion_matrix(y_test, y_pred)

    metrics = {
        "Model": model_name,
        "Resampling": resampling_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "AUC": auc
    }

    return metrics, cm


def print_confusion_matrix(cm, model_name, resampling_name):
    tn, fp, fn, tp = cm.ravel()
    print(f"\nConfusion Matrix - {model_name} ({resampling_name})")
    print("-------------------------------------------")
    print(f"           Predicted 0    Predicted 1")
    print(f"Actual 0      {tn:3d}           {fp:3d}")
    print(f"Actual 1      {fn:3d}           {tp:3d}")


# 6. RESAMPLING HELPERS (SMOTE, ADASYN…)
def apply_resampling(X_train, y_train, method="SMOTE"):
    """
    Apply the given resampling method on training data.
    method: "SMOTE", "ADASYN", "BorderlineSMOTE"
    """
    if method == "SMOTE":
        sampler = SMOTE(random_state=RANDOM_STATE)
    elif method == "ADASYN":
        sampler = ADASYN(random_state=RANDOM_STATE)
    elif method == "BorderlineSMOTE":
        sampler = BorderlineSMOTE(random_state=RANDOM_STATE)
    else:
        raise ValueError("Unknown resampling method: " + method)

    X_res, y_res = sampler.fit_resample(X_train, y_train)
    print(f"{method}: Before = {np.bincount(y_train)}, After = {np.bincount(y_res)}")
    return X_res, y_res



# 7. MAIN PIPELINE: TRAIN & EVALUATE

def main():
    # 7.1 Load data
    df = load_dataset(DATA_PATH)

    # 7.2 Split X, y
    X, y = split_features_target(df, TARGET_COL)

    # 7.3 Find numeric & categorical columns
    numeric_cols, categorical_cols = get_column_types(X)

    print("Numeric columns:", numeric_cols)
    print("Categorical columns:", categorical_cols)

    # 7.4 Build preprocessor (encoding + scaling)
    preprocessor = build_preprocessor(numeric_cols, categorical_cols)

    # 7.5 Train-test split (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )

    print("Train size:", X_train.shape, " Test size:", X_test.shape)
    print("Train class distribution:", np.bincount(y_train))
    print("Test class distribution :", np.bincount(y_test))

    # 7.6 Get models
    models = get_models()

    # To store metrics & confusion matrices for all runs
    all_metrics = []
    all_conf_mats = {}


    # A) BEFORE RESAMPLING (BASELINE)

    print("A) BASELINE (NO RESAMPLING)")
    for model_name, model in models.items():
        # Build pipeline: preprocessing + model
        clf = Pipeline(steps=[
            ("preprocess", preprocessor),
            ("model", model)
        ])

        # Fit on original imbalanced data
        clf.fit(X_train, y_train)

        # Evaluate
        metrics, cm = evaluate_model(
            clf, X_test, y_test,
            model_name=model_name,
            resampling_name="None"
        )

        all_metrics.append(metrics)
        all_conf_mats[(model_name, "None")] = cm
        print_confusion_matrix(cm, model_name, "None")


    # B) AFTER RESAMPLING
    #    1. SMOTE
    #    2. ADASYN
    #    3. Borderline-SMOTE

    resampling_methods = ["SMOTE", "ADASYN", "BorderlineSMOTE"]

    for res_method in resampling_methods:

        print(f"B) RESAMPLING WITH {res_method}")
        # NOTE: We must transform X_train first with preprocessor,
        # then resample in the transformed space.
        # So we fit_transform preprocessor only on X_train.
        X_train_processed = preprocessor.fit_transform(X_train)
        X_test_processed = preprocessor.transform(X_test)

        # Apply resampling
        X_resampled, y_resampled = apply_resampling(
            X_train_processed, y_train.to_numpy(), method=res_method
        )

        for model_name, model in models.items():
            print(f"\nTraining {model_name} with {res_method}...")

            # For models, we now skip ColumnTransformer inside Pipeline
            # because we already processed X.
            # So we fit directly on resampled, scaled, encoded data.
            clf = model
            clf.fit(X_resampled, y_resampled)

            # Evaluate (using processed test set)
            metrics, cm = evaluate_model(
                clf, X_test_processed, y_test,
                model_name=model_name,
                resampling_name=res_method
            )

            all_metrics.append(metrics)
            all_conf_mats[(model_name, res_method)] = cm
            print_confusion_matrix(cm, model_name, res_method)

    # 8. BUILD TABLES SIMILAR TO DISSERTATION

    results_df = pd.DataFrame(all_metrics)

    # Round for nicer display
    results_df["Accuracy"] = results_df["Accuracy"].round(2)
    results_df["Precision"] = results_df["Precision"].round(2)
    results_df["Recall"] = results_df["Recall"].round(2)
    results_df["F1-Score"] = results_df["F1-Score"].round(2)
    results_df["AUC"] = results_df["AUC"].round(2)


    print("TABLE: ALL MODEL RESULTS")
      if TABULATE_AVAILABLE:
        print(tabulate(results_df, headers="keys", tablefmt="github", showindex=False))
    else:
        print(results_df)

    # Example: Table like "Performance Before Resampling"
    baseline_table = results_df[results_df["Resampling"] == "None"].copy()

    print("TABLE 4.1 - BASELINE (NO RESAMPLING)")

    if TABULATE_AVAILABLE:
        print(tabulate(
            baseline_table[["Model", "Accuracy", "Precision", "Recall", "F1-Score", "AUC"]],
            headers="keys",
            tablefmt="github",
            showindex=False
        ))
    else:
        print(baseline_table[["Model", "Accuracy", "Precision", "Recall", "F1-Score", "AUC"]])

    # Example: Table after SMOTE (like 4.6 in your file)
    smote_table = results_df[results_df["Resampling"] == "SMOTE"].copy()

    print("TABLE 4.6 - AFTER SMOTE")

    if TABULATE_AVAILABLE:
        print(tabulate(
            smote_table[["Model", "Accuracy", "Precision", "Recall", "F1-Score", "AUC"]],
            headers="keys",
            tablefmt="github",
            showindex=False
        ))
    else:
        print(smote_table[["Model", "Accuracy", "Precision", "Recall", "F1-Score", "AUC"]])

    # Example: Combined comparison table like 4.11 (SMOTE vs ADASYN vs Borderline)
    comp_table = results_df[results_df["Resampling"].isin(resampling_methods)].copy()

    print("TABLE 4.11 - COMPARISON (SMOTE, ADASYN, Borderline-SMOTE)")

    if TABULATE_AVAILABLE:
        print(tabulate(
            comp_table[["Model", "Resampling", "Accuracy", "Precision", "Recall", "F1-Score", "AUC"]],
            headers="keys",
            tablefmt="github",
            showindex=False
        ))
    else:
        print(comp_table[["Model", "Resampling", "Accuracy", "Precision", "Recall", "F1-Score", "AUC"]])

    # --------------- OPTIONAL: SAVE TO CSV ---------------
    results_df.to_csv("model_results_all.csv", index=False)
    print("\nAll results saved to model_results_all.csv")


# Run everything
if __name__ == "__main__":
    main()


Dataset loaded. Shape: (303, 55)
Numeric columns: ['Age', 'Weight', 'Length', 'BMI', 'DM', 'HTN', 'Current Smoker', 'EX-Smoker', 'FH', 'BP', 'PR', 'Edema', 'Typical Chest Pain', 'Function Class', 'Q Wave', 'St Elevation', 'St Depression', 'Tinversion', 'FBS', 'CR', 'TG', 'LDL', 'HDL', 'BUN', 'ESR', 'HB', 'K', 'Na', 'WBC', 'Lymph', 'Neut', 'PLT', 'EF-TTE', 'Region RWMA']
Categorical columns: ['Sex', 'Obesity', 'CRF', 'CVA', 'Airway disease', 'Thyroid Disease', 'CHF', 'DLP', 'Weak Peripheral Pulse', 'Lung rales', 'Systolic Murmur', 'Diastolic Murmur', 'Dyspnea', 'Atypical', 'Nonanginal', 'Exertional CP', 'LowTH Ang', 'LVH', 'Poor R Progression', 'VHD']
Train size: (242, 54)  Test size: (61, 54)
Train class distribution: [ 69 173]
Test class distribution : [18 43]

A) BASELINE (NO RESAMPLING)

Confusion Matrix - RandomForest (None)
-------------------------------------------
           Predicted 0    Predicted 1
Actual 0        7            11
Actual 1        1            42

Confusion Ma